# Track 4 · 02 EEGNet — the organisers' pose regressor, on a balanced emg2pose sample

**One job.** Train the organisers' EEGNet pose baseline on the windows from
`track4_01_prepare`, check it learns, measure it on unseen people, and package it.

Self-contained: upload this one file.

---

## The recipe, from source

Model: `tracks/emg_pose/solvers/eegnet_pose.py`. braindecode EEGNet with
`final_conv_length=1`, so it emits a joint-angle vector per remaining time step; targets
standardised by one train-split mean and std, kept as buffers; Adam at 1e-3. From the
neuralbench `emg/pose` config: an **L1 loss** (the metric is a mean absolute error; on
Track 3 a squared loss pulled against the metric) and **electrode band rotation** with
offsets −1, 0, +1 on every batch (`band_offsets: [-1, 0, 1]`), which mimics the wristband
sitting one electrode round on another wrist, so the model cannot tie itself to one
placement.

Ours: a cosine learning-rate decay and seeds averaged, as on Tracks 2 and 3. And EEGNet's
output (156 steps per 5 s, ~31 Hz) is linearly upsampled to 200 Hz before the loss and at
inference. The harness stretches predictions to 2 kHz by nearest neighbour; on a real
recording, holding angles at 31 Hz costs ~0.84° against the truth, at 200 Hz ~0.12°.

## Scoring

Mean absolute angle error in **degrees**, averaged over joints and time (the harness
computes it in radians and converts). Lower is better. Measured here on the **15 validation
people**, whom the model never trains on: the same kind of test as the warm-up's 20 unseen
people, though the scored test also holds out movement types. Reference points on the
warm-up: our all-zeros entry 22.67°, the organisers' mean-pose floor 21.60°, the published
NeuroPose reference 17.5 ± 1.5°, the leader 12.57° when last checked.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

EPOCHS = 40
LR = 1e-3
LR_SCHEDULE = 'cosine'    # 'cosine' or 'constant'
BATCH = 64                # emg/pose config
F1 = 8                    # EEGNet temporal filters (default 8; Track 2 gained from wider)
DROPOUT = 0.25            # EEGNet default
WEIGHT_DECAY = 0.0
ROTATE = True             # task config's band rotation, offsets -1/0/+1, every batch
N_SEEDS = 3
KEEP = 'best'             # 'best' val epoch (15 unseen people) or 'last'

RUN_CONTROL = True
MAKE_ZIP = True
SEED = 33

RUN_TAG = f'f{F1}_d{DROPOUT}_b{BATCH}_{LR_SCHEDULE}_rot{int(ROTATE)}_{KEEP}_x{N_SEEDS}'
print(f'NOTEBOOK VERSION: track4_02 v1   settings {RUN_TAG}   epochs {EPOCHS}')

In [ ]:
%pip install -q 'braindecode>=1.8'

import importlib
for mod in ('braindecode', 'torch', 'numpy'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track4'
DATA = BASE / 'emg2pose'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=3):
    """np.load that survives the Drive mount dropping mid-read (Errno 107)."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        except OSError as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed ({e.strerror}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
ST = man['stored']
parts = {g: ([], [], []) for g in ('train', 'val')}
t0 = time.time()
for f in sorted((DATA / 'windows').glob('sub-*.npz')):
    z = load_npz(f)
    if 'X' not in z:
        continue
    info = json.loads(str(z['info']))
    p = parts[info['group']]
    p[0].append(z['X']); p[1].append(z['Y']); p[2].append(np.full(len(z['X']), info['sub']))
X, Y, P = {}, {}, {}
for g, (xs, ys, ps) in parts.items():
    X[g], Y[g], P[g] = np.concatenate(xs), np.concatenate(ys), np.concatenate(ps)   # float16
assert not set(P['train']) & set(P['val'])
N_CHANS, N_TIMES = X['train'].shape[1:]
N_JOINTS, OUT_STEPS = Y['train'].shape[1:]
EMG_SFREQ, NATIVE_SFREQ = ST['emg_sfreq'], man['harness']['native_sfreq']
print(f'train {len(X["train"]):,} windows / {len(set(P["train"]))} people, val {len(X["val"]):,} / '
      f'{len(set(P["val"]))} people   ({time.time() - t0:.0f}s)')
print(f'EMG {N_CHANS} ch x {N_TIMES} at {EMG_SFREQ:g} Hz (uV), angles {N_JOINTS} x {OUT_STEPS} '
      f'at {ST["target_sfreq"]:g} Hz (rad), device {DEVICE}')

deg = lambda r: float(np.degrees(r))


def mae_deg(pred, y):
    return deg(np.abs(pred - y.astype('float32')).mean())


mean_pose = Y['train'].astype('float32').mean(axis=(0, 2), keepdims=True)
floor_val = mae_deg(np.broadcast_to(mean_pose, Y['val'].shape), Y['val'])
print(f'mean-pose prediction on val: {floor_val:.2f} deg   (organisers\' floor on the warm-up 21.60)')

In [ ]:
from braindecode.augmentation.functional import band_rotation


class DensePose(torch.nn.Module):
    """Same as the submission's DensePose (solvers/track4_pose_eegnet.py)."""

    def __init__(self, n_chans, n_joints, n_times, out_steps, f1=8, dropout=0.25):
        super().__init__()
        from braindecode.models import EEGNet
        self.out_steps = out_steps
        self.net = EEGNet(n_chans=n_chans, n_outputs=n_joints, n_times=n_times,
                          final_conv_length=1, F1=f1, drop_prob=dropout)
        self.register_buffer('y_mean', torch.zeros(1))
        self.register_buffer('y_std', torch.ones(1))

    def forward(self, X):
        out = self.net(X)
        out = out[..., None] if out.ndim == 2 else out
        return F.interpolate(out, size=self.out_steps, mode='linear', align_corners=False)

    def to_radians(self, out):
        return out * self.y_std + self.y_mean


def train(Xtr, Ytr, Xv, Yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    aug_rng = np.random.RandomState(seed)
    net = DensePose(N_CHANS, N_JOINTS, N_TIMES, OUT_STEPS, F1, DROPOUT).to(DEVICE)
    yt = Ytr.astype('float32')
    net.y_mean.fill_(float(yt.mean())); net.y_std.fill_(float(yt.std()))
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    Xtr_t, Ytr_t = torch.from_numpy(Xtr), torch.from_numpy(Ytr)

    @torch.no_grad()
    def infer(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net.to_radians(net(Xa[i:i + 128].to(DEVICE).float())).cpu()
                          for i in range(0, len(Xa), 128)]).numpy()

    if not quiet:
        print(f'{label}: EEGNet-pose {sum(p.numel() for p in net.parameters()):,} parameters, '
              f'{len(Xtr):,} windows, rotation {ROTATE}, seed {seed}')
    best, best_state, hist = np.inf, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(Xtr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            xb = Xtr_t[b].to(DEVICE).float()
            yb = Ytr_t[b].to(DEVICE).float()
            if ROTATE:
                xb, yb = band_rotation(xb, yb, num_bands=1, electrodes_per_band=N_CHANS,
                                       band_offsets=(-1, 0, 1), random_state=aug_rng)
            opt.zero_grad()
            loss = F.l1_loss(net(xb), (yb - net.y_mean) / net.y_std)
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        v = mae_deg(infer(Xv), Yv)
        hist.append(dict(epoch=ep, train_l1=tot / len(order), val_mae_deg=v))
        better = v < best
        if better:
            best = v
            best_state = {k: t.detach().cpu().clone() for k, t in net.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train L1 {tot / len(order):.4f}  val MAE {v:6.2f} deg  '
                  f'lr {lr_now:.1e}{"  *" if better else ""}  {time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state)
    return net, hist, infer


print('ready')

## Step 1 — positive control

A random mix of the joint angles is planted into a copy of the EMG, and the same loop
runs on it. If this does not learn, the loop is broken and Step 2 means nothing.

In [ ]:
if RUN_CONTROL:
    mix = rng.standard_normal((N_CHANS, N_JOINTS)).astype('float32')

    def plant(Xa, Ya):
        up = np.repeat(Ya.astype('float32'), N_TIMES // OUT_STEPS, axis=-1)[..., :N_TIMES]
        sig = np.einsum('cj,bjt->bct', mix, up - up.mean(-1, keepdims=True))
        sig *= Xa.astype('float32').std() / (sig.std() + 1e-9)
        return (Xa.astype('float32') + 2.0 * sig).astype('float16')

    sub = rng.choice(len(X['train']), min(3000, len(X['train'])), replace=False)
    _, hist_c, _ = train(plant(X['train'][sub], Y['train'][sub]), Y['train'][sub],
                         plant(X['val'], Y['val']), Y['val'], 8, 'control', quiet=True)
    best_c = min(h['val_mae_deg'] for h in hist_c)
    print(f'control val MAE: epoch 1 {hist_c[0]["val_mae_deg"]:.2f} -> best {best_c:.2f} deg   '
          f'(mean pose {floor_val:.2f})')
    print('CONTROL PASSES: the loop learns a planted signal' if best_c < 0.6 * floor_val else
          'CONTROL FAILS: the loop cannot learn a planted signal; fix before Step 2')
else:
    print('control skipped')

## Step 2 — the real run

What to look for: validation error falling clearly below the mean-pose floor and below its
own null (the same predictions shuffled across windows), with the best epoch after the first.

In [ ]:
members = []
for i in range(N_SEEDS):
    net_i, hist_i, infer_i = train(X['train'], Y['train'], X['val'], Y['val'], EPOCHS,
                                   f'real, seed {i + 1}/{N_SEEDS}', seed=SEED + i)
    kept = (min(hist_i, key=lambda h: h['val_mae_deg']) if KEEP == 'best' else hist_i[-1])
    members.append(dict(net=net_i, hist=hist_i, infer=infer_i, kept=kept))
    print(f'  -> seed {i + 1}: kept epoch {kept["epoch"]} ({KEEP}), val MAE {kept["val_mae_deg"]:.2f} deg\n')


def predict(Xa):
    return np.mean([m['infer'](Xa) for m in members], axis=0)


pv = predict(X['val'])
v_mae = mae_deg(pv, Y['val'])
null = np.array([mae_deg(pv[rng.permutation(len(pv))], Y['val']) for _ in range(50)])
hist = members[0]['hist']
TRENDING = bool(v_mae < null.min() and v_mae < floor_val and members[0]['kept']['epoch'] > 1)
seed_v = [m['kept']['val_mae_deg'] for m in members]
per_joint = np.degrees(np.abs(pv - Y['val'].astype('float32')).mean(axis=(0, 2)))
per_person = {s: mae_deg(pv[P['val'] == s], Y['val'][P['val'] == s]) for s in sorted(set(P['val']))}
pp = np.array(list(per_person.values()))

print(f'val (15 unseen people)   mean of {N_SEEDS} seeds: {v_mae:.2f} deg')
print(f'  single seeds: {", ".join(f"{v:.2f}" for v in seed_v)}')
print(f'  null (predictions shuffled across windows): {null.mean():.2f} +- {null.std():.2f} deg')
print(f'  mean pose: {floor_val:.2f} deg    per person: best {pp.min():.2f}  median {np.median(pp):.2f}  '
      f'worst {pp.max():.2f}')
print(f'  per joint: easiest {per_joint.min():.1f}  hardest {per_joint.max():.1f} deg')
print('  warm-up references: zeros 22.67, mean pose 21.60, NeuroPose 17.5, leader 12.57 (deg)')
print('Trending: beats the mean pose and its null after epoch 1.' if TRENDING else
      'Not trending: does not clear the mean pose and its null.')

for i, m in enumerate(members):
    torch.save(m['net'].state_dict(), WEIGHTS / f'pose_eegnet_{RUN_TAG}_s{i}.pt')
(WEIGHTS / f'pose_eegnet_{RUN_TAG}.json').write_text(json.dumps(dict(
    run_tag=RUN_TAG, f1=F1, dropout=DROPOUT, batch=BATCH, rotate=ROTATE, keep=KEEP,
    lr_schedule=LR_SCHEDULE, n_seeds=N_SEEDS, val_mae_deg=v_mae, per_person=per_person,
    members=[dict(kept_epoch=m['kept']['epoch'], history=m['hist']) for m in members]), indent=1))
print(f'\nsaved {N_SEEDS} members -> {WEIGHTS / f"pose_eegnet_{RUN_TAG}_s*.pt"}')

## Step 3 — package for submission

The solver receives raw 2 kHz EMG in volts and must turn it into exactly what training
saw. The checks feed it raw-style input and compare against this notebook's own pipeline
(mean removal, the same resampler, microvolts), so a mismatch shows up here, not on the board.

In [ ]:
# Source of truth is solvers/track4_pose_eegnet.py in the repo.
SUBMISSION_PY = "\"\"\"Track 04 submission: EEGNet regressing 20 joint-angle trajectories from wrist EMG.\n\nThe model follows the organisers' `eegnet_pose.py` baseline: braindecode EEGNet with\n`final_conv_length=1`, so it emits one joint-angle vector per remaining time step, and\ntargets standardised by one train-split mean and std kept as buffers. It is trained\nwith an L1 loss (the metric is a mean absolute error) and the task config's electrode\nband rotation, in `track4_02_eegnet.ipynb`.\n\nInput handling repeats the prepare notebook exactly, so training and scoring see the\nsame data: per-window, per-channel mean removal, resampling from the harness's 2 kHz\nto 1 kHz with the same rfft function, and volts to microvolts. EEGNet's output (about\n31 steps per second) is linearly upsampled to 200 Hz; the harness then stretches that\nto 2 kHz by nearest neighbour. Electrodes are matched by name, and a zip may hold\nseveral seeds whose predictions are averaged.\n\"\"\"\n\nimport json\n\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import EEGNet\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation; the prepare notebook uses the same function.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass DensePose(nn.Module):\n    \"\"\"EEGNet emitting a standardised joint-angle sequence, upsampled to out_steps.\"\"\"\n\n    def __init__(self, n_chans, n_joints, n_times, out_steps, f1=8, dropout=0.25):\n        super().__init__()\n        self.out_steps = out_steps\n        self.net = EEGNet(\n            n_chans=n_chans, n_outputs=n_joints, n_times=n_times,\n            final_conv_length=1, F1=f1, drop_prob=dropout,\n        )\n        self.register_buffer(\"y_mean\", torch.zeros(1))\n        self.register_buffer(\"y_std\", torch.ones(1))\n\n    def forward(self, X):                          # standardised (B, J, out_steps)\n        out = self.net(X)\n        out = out[..., None] if out.ndim == 2 else out\n        return F.interpolate(out, size=self.out_steps, mode=\"linear\", align_corners=False)\n\n    def to_radians(self, out):\n        return out * self.y_std + self.y_mean\n\n\nclass ChannelMatchedPose:\n\n    def __init__(self, cfg, in_names, in_sfreq, n_joints, device):\n        if n_joints not in (None, cfg[\"n_joints\"]):\n            raise RuntimeError(f\"trained for {cfg['n_joints']} joints, harness asks for {n_joints}\")\n        self.device = device\n        self.cfg = cfg\n        self.in_sfreq = float(in_sfreq)\n        self.nets = [\n            DensePose(len(cfg[\"channels\"]), cfg[\"n_joints\"], cfg[\"n_times\"],\n                      cfg[\"out_steps\"], cfg[\"f1\"], cfg[\"dropout\"]).to(device).eval()\n            for _ in cfg[\"weight_files\"]\n        ]\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg[\"channels\"]]\n        missing = [n for n, i in zip(cfg[\"channels\"], self.src) if i < 0]\n        print(\n            f\"[EEGNet-pose] matched {len(self.src) - len(missing)}/{len(self.src)} \"\n            f\"training channels by name among {len(in_names)} inputs; \"\n            f\"zero-filled: {missing if missing else 'none'}\"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(f\"no training channel found in the input: {list(in_names)[:8]}\")\n\n    @torch.no_grad()\n    def predict(self, X):\n        cfg = self.cfg\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        out = out - out.mean(-1, keepdim=True)                       # as in prepare\n        n_out = int(round(out.shape[-1] * cfg[\"sfreq\"] / self.in_sfreq))\n        out = resample_fft(out, n_out) * cfg[\"emg_scale\"]\n        n = int(cfg[\"n_times\"])\n        out = out[..., :n] if out.shape[-1] >= n else F.pad(out, (0, n - out.shape[-1]))\n        preds = [net.to_radians(net(out)) for net in self.nets]\n        return torch.stack(preds).mean(0)                           # (B, J, out_steps) radians\n\n\nclass Solver(CompetSolver):\n\n    name = \"EEGNet-pose\"\n\n    requirements = [\"pip::braindecode\"]\n\n    def load_model(self, meta):\n        sub = meta[\"submission_dir\"]\n        cfg = json.loads((sub / \"pose_eegnet_config.json\").read_text())\n        in_names = meta.get(\"ch_names\") or [c[\"ch_name\"] for c in meta[\"chs_info\"]]\n        model = ChannelMatchedPose(cfg, in_names, meta[\"sfreq\"], meta.get(\"n_joints\"),\n                                   meta[\"device\"])\n        for net, name in zip(model.nets, cfg[\"weight_files\"]):\n            net.load_state_dict(torch.load(sub / name, map_location=meta[\"device\"],\n                                           weights_only=True))\n        return model\n"

In [ ]:
def resample_fft(x, n_out):
    """Same function as the prepare notebook and the solver."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


if MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the run did not clear the mean pose and its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / f'track4_eegnet_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    files = [f'weights_{i}.pt' for i in range(len(members))]
    for f_, m in zip(files, members):
        torch.save({k: v.cpu() for k, v in m['net'].state_dict().items()}, out / f_)
    CH = [f'emg{i}' for i in range(N_CHANS)]
    (out / 'pose_eegnet_config.json').write_text(json.dumps(dict(
        channels=CH, n_joints=int(N_JOINTS), n_times=int(N_TIMES), out_steps=int(OUT_STEPS),
        sfreq=EMG_SFREQ, emg_scale=ST['emg_scale'], f1=F1, dropout=DROPOUT, weight_files=files,
        trained_on=f'emg2pose balanced sample, {len(set(P["train"]))} people (scored test people excluded)',
        val_mae_deg=v_mae), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, Xa, n_joints=N_JOINTS):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None, 'sfreq': NATIVE_SFREQ,
                                     'submission_dir': out, 'device': DEVICE, 'n_joints': n_joints})
        return m.predict(torch.tensor(Xa)).cpu().numpy()

    # Raw-style input: 2 kHz volts with a large constant offset, like unfiltered EMG.
    n_native = int(round(N_TIMES * NATIVE_SFREQ / EMG_SFREQ))
    raw_x = (rng.standard_normal((8, N_CHANS, n_native)) * 3e-5 + 0.05).astype('float32')
    mine = resample_fft(torch.from_numpy(raw_x - raw_x.mean(-1, keepdims=True)), N_TIMES).numpy() * ST['emg_scale']
    expect = predict(mine.astype('float16'))
    base = run(CH, raw_x)
    perm = rng.permutation(N_CHANS)
    shuffled = run([CH[i] for i in perm], raw_x[:, perm])
    no_offset = run(CH, raw_x - 0.05)
    wide = run(CH + ['joint0'], np.concatenate([raw_x, rng.standard_normal((8, 1, n_native)).astype('float32')], 1))
    try:
        run(CH, raw_x[:2], n_joints=21); wrong = False
    except RuntimeError:
        wrong = True
    checks = [
        ('shape and range', base.shape == (8, N_JOINTS, OUT_STEPS) and np.isfinite(base).all()
         and np.abs(base).max() < 2 * np.pi, f'{base.shape}, |angle| <= {np.abs(base).max():.2f} rad'),
        ('same pipeline as training', np.allclose(base, expect, atol=2e-3),
         f'max |diff| {np.abs(base - expect).max():.1e} rad (training data is float16)'),
        ('offset removed', np.allclose(base, no_offset, atol=1e-4), 'a 50 mV offset on every channel changes nothing'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-5), 'electrodes shuffled'),
        ('extra channels ignored', np.allclose(base, wide, atol=1e-5), 'a joint channel added'),
        ('wrong joint count raises', wrong, 'asked for 21 joints: refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench (Track 4).')
    else:
        print('\nNot zipping: a check failed.')